# Qian — LoRA versus DoRA at 448px
Compare adaptation methods using crop/resolution **arm E**: 75% random crop + 25% whole-image/padding, prototype cosine head, no VPT. The official augreg2 ViT-B/16 and 10,000-way head stay the same.

Parameter preflight chooses the largest shared rank below **5% trainable backbone parameters**, classifier excluded. If DoRA r24 exceeds the cap, train **LoRA r24 (reference), LoRA at the shared rank, and DoRA at the shared rank**. The shared-rank pair isolates the method change; DoRA has additional magnitude parameters.

Use one **A100 runtime** for all arms. Default: seed 42, **12 training minutes per arm**, plus prototype extraction, evaluation and saving. Repeat a promising comparison with training seed 43 while keeping split/prototype seed 42 fixed. Existing crop/resolution outputs are kept separately.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/qq_lora_dora_repo"
BRANCH = "qq_lora_ablation"  # Commit and push these changes before running in Colab.
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
EXPERIMENT_NAME = "qq_lora_dora_448_c1000_v1"
CONFIG = "configs/qq_lora_dora_ablation.yaml"
MINUTES_PER_ARM = 12
NUM_CLASSES_SUBSET = 1000
SEEDS = [42]  # After reviewing seed 42, use [42, 43] to reuse it and add seed 43.
NUM_WORKERS = 8
DOWNLOAD_DATA = True
# Keep this name/settings to reuse completed arms. Use a new name after code,
# hardware or recipe changes, or an interrupted arm. Partial arms are not resumed.


## 1. Connect to GitHub
Same hidden-token connection as your training launcher. Checkout/pull `qq_lora_ablation`; this does not merge it with main.


In [ ]:
import sys
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


## 2. Install packages and check the GPU


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime; A100 is recommended."
print("GPU:", torch.cuda.get_device_name())
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if 'A100' not in torch.cuda.get_device_name():
    print("For comparable timing, run all arms on the same A100. Do not mix hardware between arms.")


## 3. Mount Drive
Use the same `CS604_RUNS` shared-folder shortcut as before. Dataset images stay on temporary Colab storage; results and checkpoint files go to Drive.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
base_out = Path(DRIVE_OUT)
if not base_out.is_dir():
    raise FileNotFoundError(f"{DRIVE_OUT} not found. Add the shared CS604_RUNS folder as a shortcut in My Drive.")
EXPERIMENT_DIR = base_out / EXPERIMENT_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Experiment:", EXPERIMENT_DIR)


## 4. Download or reuse extracted images
The full download/extraction can be slow and needs room for archives plus images. Check disk space first. Set `DOWNLOAD_DATA=False` to use an existing extract. Reuse is based on a small path check; it does not certify every file.


In [ ]:
# Stream subprocess output into the notebook, and stop on failure.
def run_live(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
        process.wait()
        raise


import shutil
root = Path(DATA_DIR)
root.mkdir(parents=True, exist_ok=True)
print(f"Free Colab disk: {shutil.disk_usage(root).free / 2**30:.1f} GiB")
def extracted_training_available():
    listing = root / 'train.txt'
    if not listing.is_file(): return False
    paths = []
    with listing.open() as handle:
        for line in handle:
            if line.strip(): paths.append(line.split()[0])
            if len(paths) == 20: break
    return bool(paths) and all((root / path).is_file() for path in paths)
if DOWNLOAD_DATA and not extracted_training_available():
    run_live([sys.executable, '-u', 'scripts/download_data.py', '--dest', DATA_DIR])
else:
    print('Using existing dataset:', root)
assert (root / 'train.txt').is_file(), 'Set DATA_DIR to the extracted training dataset.'


## 5. Review settings and exact parameter counts
The preflight creates an untrained CPU model to count parameters; it does not download pretrained weights or train anything. At 448px, DoRA r24 is expected to exceed 5%, so the shared rank should be 23. Each arm's LoRA alpha equals its rank, keeping the scaling at 1.

All arms start independently from pretrained weights. They share the same 1,000 classes, five holdout images/class, fitting-only prototypes, optimizer, learning rates, effective batch size and inference settings. The split algorithm and seed match the earlier crop/resolution experiment; no previous trained checkpoint or all-data prototypes are loaded.

Evaluation is **crop 0.9, no flip or zoom**. Time includes training data loading and augmentation, with 5% time warmup and cosine decay; validation, setup, prototype building and saving are excluded. If any arm runs out of memory, use a new experiment name and reduce the microbatch for **all** arms while increasing accumulation to keep effective batch 256. Make this change in a config committed to the branch.


In [ ]:
from src.utils import load_config
cfg = load_config(CONFIG)
print("Backbone:", cfg.model_name)
print("Resolution / padding probability:", cfg.img_size, cfg.full_image_probability)
print("Head:", cfg.head_type, "| Extra trainable parts:", cfg.unfreeze_extra)
print("Adapter / head LR:", cfg.lr, cfg.head_lr)
print("Batch / accumulation:", cfg.batch_size, cfg.grad_accum_steps)
print("Training seeds:", SEEDS, "| Fixed split / prototype seed:", cfg.split_seed, cfg.prototype_seed)
print("Subset classes:", NUM_CLASSES_SUBSET, "| Holdout per class:", cfg.ablation_holdout)
run_live([sys.executable, '-u', '-m', 'src.qq_lora_dora_ablation',
          '--config', CONFIG, '--preflight-only', '--minutes', str(MINUTES_PER_ARM),
          '--seeds', *map(str, SEEDS), '--classes', str(NUM_CLASSES_SUBSET)])


## 6. Run the comparison
The runner saves each completed arm immediately and checks code, settings, split, package versions and hardware before reusing it. Run seed 42 first. If the shared-rank DoRA result looks promising, change `SEEDS` to `[42, 43]` in Settings, rerun Settings and this cell. Keep the same output name to reuse completed seed-42 arms.

The main score is **final holdout accuracy after equal training minutes**. Best-epoch accuracy is recorded for context. This screen does not automatically launch full-data training or generate a submission.


In [ ]:
command = [sys.executable, '-u', '-m', 'src.qq_lora_dora_ablation',
           '--config', CONFIG, '--data_root', DATA_DIR, '--output_dir', str(EXPERIMENT_DIR),
           '--minutes', str(MINUTES_PER_ARM), '--classes', str(NUM_CLASSES_SUBSET),
           '--seeds', *map(str, SEEDS), '--workers', str(NUM_WORKERS)]
run_live(command)


## 7. Compare accuracy, throughput and memory
`results.csv` contains all arms requested in `SEEDS`. `comparison.json` compares the shared-rank pair per seed, including final accuracy difference, speed ratio and metrics at common completed epochs. A 0.3-point gain is a screening threshold, not statistical significance. A second seed checks training variability on the same fixed split.

Common-epoch comparisons exclude partial epochs. Because the primary experiment uses a time-based schedule, the methods can be at different learning rates at a matching epoch; these metrics do not isolate adaptation quality from schedule effects.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
results = pd.read_csv(EXPERIMENT_DIR / 'results.csv')
display(results[['seed', 'arm', 'rank', 'dora', 'final_val_acc', 'best_val_acc',
                 'final_val_loss', 'training_minutes', 'epochs', 'images_per_second',
                 'peak_vram_gib', 'adaptation_pct']])
comparison = json.loads((EXPERIMENT_DIR / 'comparison.json').read_text())
for row in comparison['comparisons']:
    print(f"Seed {row['seed']}, shared rank {row['shared_rank']}: "
          f"DoRA minus LoRA {row['delta_pp']:+.2f} percentage points; "
          f"DoRA/LoRA throughput {row['speed_ratio']:.3f}x")
    display(pd.DataFrame(row['matched_complete_epochs']))
print(comparison['note'])
print('Clears the screening threshold across at least two seeds:', comparison['promising_across_seeds'])
labels = results['seed'].astype(str) + '/' + results['arm']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, field, title in zip(axes, ['final_val_acc', 'images_per_second', 'peak_vram_gib'],
                           ['Final holdout accuracy (%)', 'Training images / second', 'Peak memory (GiB)']):
    values = results[field] * 100 if field == 'final_val_acc' else results[field]
    ax.bar(labels, values)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / 'comparison.png', dpi=150)
plt.show()


## 8. Training curves and next decision
Review the accuracy gain together with throughput, completed epochs and peak memory. DoRA may train more slowly because it also computes adapted-weight norms. Repeat a promising result with seed 43 before deciding on a full-data run; a tiny or inconsistent gain is inconclusive.

Outputs include resolved configs, best/last checkpoints, epoch/step logs, summary CSV/JSON, plots and copied split manifests. Full-data training would require fresh pretrained weights, fresh all-data prototypes and a separate 144-minute budget. Subset accuracy is not comparable to CodaBench accuracy.


In [ ]:
for row in results.itertuples(index=False):
    run_live([sys.executable, 'scripts/plot_curves.py', '--run_dir', row.run_dir,
              '--out_dir', str(EXPERIMENT_DIR / f'seed{row.seed}' / row.arm / 'plots')])
print('Saved results:', EXPERIMENT_DIR)
